# Fully Automated Bilingual Dataset Creator (Google ASR)

This notebook automates the entire pipeline for creating a Malayalam-English bilingual dataset from a YouTube **playlist**.

**Key Features:**
1.  **Playlist Processing**: Automatically fetches all videos from a YouTube playlist.
2.  **Progress Tracking**: Uses an Excel file (`playlist_progress.xlsx`) to track completed videos, allowing you to stop and resume.
3.  **Music Removal**: Uses Demucs to separate vocals from background music for cleaner audio.
4.  **Intelligent 8-Second Chunks**: Splits audio on silent pauses, ensuring no chunk exceeds 8 seconds.
5.  **Google ASR**: Transcribes Malayalam audio using Google's Speech Recognition.
6.  **Gemini Translation**: Translates the Malayalam text to English.
7.  **Real-time Cleanup**: Automatically deletes audio chunks if transcription fails.
8.  **Timed Pauses**: After every 10 videos, the script pauses for 5 minutes, giving you the option to halt the process.

In [ ]:
# --- 1. Setup and Installation ---
from google.colab import drive
import os
import torch

drive.mount('/content/drive')

print("🚀 Installing and updating libraries...")
!pip install --upgrade --force-reinstall "yt-dlp"
!pip install -q pydub speechrecognition demucs pandas openpyxl
!pip install -q google-generativeai

print("✅ Setup complete!")

Mounted at /content/drive
🚀 Installing and updating libraries...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 7.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 7.1 MB/s eta 0:00:00
✅ Set

In [ ]:
# --- 2. Configuration ---
import google.generativeai as genai
from google.colab import userdata

# ⬅️ PASTE YOUR YOUTUBE PLAYLIST URL HERE
PLAYLIST_URL = "https://youtube.com/playlist?list=PL4277B25E85866988&si=yD1_zZxcjXJE0xqq" # Change this URL

# --- Configure Gemini API Key ---
try:
    GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY_1') # Assumes key is stored in Colab Secrets
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini API configured successfully.")
except Exception as e:
    print("🚨 Could not configure Gemini API. Please set your API key in Colab Secrets.")

# --- Define Paths ---
BASE_DRIVE_PATH = "/content/drive/MyDrive/Automated_Malayalam_Dataset"
AUDIO_CHUNKS_FOLDER = os.path.join(BASE_DRIVE_PATH, "audio")
METADATA_FILE_PATH = os.path.join(BASE_DRIVE_PATH, "metadata.jsonl")
RAW_AUDIO_PATH = os.path.join(BASE_DRIVE_PATH, "raw_downloaded_audio.mp3")
DEMUCS_OUTPUT_DIR = os.path.join(BASE_DRIVE_PATH, "separated_audio")
# New path for the Excel tracking file
TRACKING_FILE_PATH = os.path.join(BASE_DRIVE_PATH, "playlist_progress.xlsx")


os.makedirs(AUDIO_CHUNKS_FOLDER, exist_ok=True)
os.makedirs(DEMUCS_OUTPUT_DIR, exist_ok=True)

print(f"✅ All files will be saved in: {BASE_DRIVE_PATH}")
print(f"✅ Progress will be tracked in: {TRACKING_FILE_PATH}")

✅ Gemini API configured successfully.
✅ All files will be saved in: /content/drive/MyDrive/Automated_Malayalam_Dataset
✅ Progress will be tracked in: /content/drive/MyDrive/Automated_Malayalam_Dataset/playlist_progress.xlsx


In [ ]:
# --- 3. Audio Processing & Music Separation Functions ---
import yt_dlp
import re
from pydub import AudioSegment
from pydub.silence import split_on_silence
from tqdm.notebook import tqdm

def download_audio_from_youtube(url, output_path):
    """Downloads audio from a YouTube URL as an MP3 file."""
    if not re.match(r'^https?://(www\.)?(youtube\.com|youtu\.be)/.+', url):
        print(f"🚨 Invalid YouTube URL: {url}")
        return None

    print(f"Downloading audio from {url}...")
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3', 'preferredquality': '192'}],
        'outtmpl': output_path.replace('.mp3', ''),
        'retries': 5,
        'verbose': False, # Set to False for cleaner loop output
        'quiet': True,
        'cookiefile': os.path.join(BASE_DRIVE_PATH, "cookies.txt"),
        'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        print(f"✅ Audio downloaded successfully to {output_path}")
        return output_path
    except Exception as e:
        print(f"🚨 Unexpected error downloading audio: {e}")
        return None

def separate_vocals_with_demucs(input_audio_path, output_dir):
    print("🎶 Separating vocals from music with Demucs... (This may take a few minutes)")
    !demucs --two-stems=vocals -o "{output_dir}" "{input_audio_path}"
    filename_base = os.path.splitext(os.path.basename(input_audio_path))[0]
    vocals_path = os.path.join(output_dir, "htdemucs", filename_base, "vocals.wav")
    if os.path.exists(vocals_path):
        print(f"✅ Vocals separated successfully. Clean audio at: {vocals_path}")
        return vocals_path
    else:
        print(f"🚨 Error: Could not find the separated vocals file.")
        return None

def trim_audio(file_path, trim_duration_seconds):
    """Trims a specified duration from the end of an audio file."""
    print(f"✂️ Trimming last {trim_duration_seconds} seconds from the audio...")
    try:
        audio = AudioSegment.from_file(file_path)
        duration_ms = len(audio)
        trim_ms = trim_duration_seconds * 1000

        if duration_ms > trim_ms:
            trimmed_audio = audio[:-trim_ms]
            trimmed_audio.export(file_path, format="mp3")
            print(f"✅ Audio trimmed successfully. New duration: {len(trimmed_audio)/1000.0:.2f} seconds.")
        else:
            print("Audio is shorter than the trim duration, no trimming was performed.")
        return file_path
    except Exception as e:
        print(f"🚨 Error trimming audio: {e}")
        return None

def preprocess_and_split_audio(input_wav_path, output_folder, max_length_ms=8000):
    try:
        start_index = 1
        if os.path.exists(output_folder):
            max_num = 0
            for f in os.listdir(output_folder):
                match = re.search(r'malayalam_speech_(\d+).wav', f)
                if match:
                    num = int(match.group(1))
                    if num > max_num: max_num = num
            start_index = max_num + 1
        print(f"New audio files will be numbered starting from {start_index}.")

        audio = AudioSegment.from_wav(input_wav_path)
        audio = audio.set_frame_rate(16000).set_channels(1)

        print("🤫 Splitting audio based on silence...")
        natural_chunks = split_on_silence(
            audio, min_silence_len=700, silence_thresh=audio.dBFS - 14, keep_silence=300
        )

        final_chunks = []
        for chunk in natural_chunks:
            if len(chunk) > max_length_ms:
                for i in range(0, len(chunk), max_length_ms):
                    final_chunks.append(chunk[i:i + max_length_ms])
            else:
                final_chunks.append(chunk)

        if not final_chunks: return []

        chunk_paths = []
        print(f"✅ Found {len(final_chunks)} potential chunks (max {max_length_ms/1000}s). Saving now...")
        for i, chunk in enumerate(tqdm(final_chunks, desc="Saving Chunks"), start=start_index):
            if len(chunk) < 2000: continue
            chunk_name = f"malayalam_speech_{i}.wav"
            chunk_path = os.path.join(output_folder, chunk_name)
            chunk.export(chunk_path, format="wav")
            chunk_paths.append(chunk_path)

        print(f"✅ Saved {len(chunk_paths)} audio chunks to {output_folder}")
        return chunk_paths
    except Exception as e:
        print(f"🚨 Error splitting audio: {e}")
        return []

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
# --- 4. Automation Helper Functions ---
import pandas as pd
import yt_dlp
import sys
import select
import time

def get_playlist_video_urls(playlist_url):
    """Extracts all individual video URLs from a YouTube playlist."""
    print(f"Fetching video URLs from playlist: {playlist_url}...")
    ydl_opts = {
        'extract_flat': True,
        'quiet': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            result = ydl.extract_info(playlist_url, download=False)
            if 'entries' in result:
                urls = [entry['url'] for entry in result['entries'] if entry]
                print(f"✅ Found {len(urls)} videos in the playlist.")
                return urls
            else:
                print("🚨 No videos found in the playlist.")
                return []
        except Exception as e:
            print(f"🚨 Could not extract playlist info: {e}")
            return []

def user_halt_check():
    """Pauses for 5 minutes and checks for user input to stop."""
    print("\n--- ⏸️ PAUSING ---")
    print("Processed 5 videos. Pausing for 1 minutes.")
    print("To stop the entire process, type 'stop' and press Enter.")
    print("Otherwise, processing will resume automatically.")

    try:
        for i in range(60, 0, -1):
            print(f"\rResuming in {i//60} minutes and {i%60} seconds... ", end="")
            # Check for input for 1 second without blocking
            ready, _, _ = select.select([sys.stdin], [], [], 1)
            if ready:
                user_input = sys.stdin.readline().strip().lower()
                if user_input == 'stop':
                    print("\n🛑 User requested to halt. Stopping dataset creation.")
                    return True # Indicates user wants to halt
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n🛑 Process interrupted by user. Halting.")
        return True

    print("\n\n▶️ Resuming processing...")
    return False # Indicates to continue

In [ ]:
# --- 5. Load Models and Recognizer ---
import speech_recognition as sr

# --- Initialize Speech Recognizer ---
print("🎙️ Initializing Google Speech Recognizer...")
recognizer = sr.Recognizer()
print("✅ Recognizer ready.")

# --- Load Gemini Model ---
print("🧠 Loading Gemini model...")
gemini_model = genai.GenerativeModel('gemini-2.0-flash')
print("✅ Gemini model loaded successfully!")

🎙️ Initializing Google Speech Recognizer...
✅ Recognizer ready.
🧠 Loading Gemini model...
✅ Gemini model loaded successfully!


In [ ]:
# --- 6. Main Pipeline (Google ASR with Real-time Cleanup) ---
import json

def create_dataset_pipeline(audio_chunk_paths, metadata_output_path):
    dataset_records = []
    print("\n🤖 Starting transcription and translation pipeline...")

    for audio_path in tqdm(audio_chunk_paths, desc="Processing Files"):
        malayalam_text = ""
        try:
            with sr.AudioFile(audio_path) as source:
                audio_data = recognizer.record(source)
                malayalam_text = recognizer.recognize_google(audio_data, language='ml-IN').strip()
        except (sr.UnknownValueError, sr.RequestError) as e:
            print(f"File: {os.path.basename(audio_path)} -> Transcription failed. Deleting file.")
            try:
                os.remove(audio_path)
            except OSError as del_e:
                print(f"  - Error deleting file: {del_e}")
            continue

        if malayalam_text:
            try:
                prompt = f"Translate the following Malayalam sentence to English. Provide only the direct translation.\n\nMalayalam: {malayalam_text}\n\nEnglish:"
                response = gemini_model.generate_content(prompt)
                english_text = response.text.strip()
            except Exception as e:
                english_text = f"Translation failed: {e}"
                print(f"Translation failed for '{malayalam_text}': {e}")
        else:
            continue

        relative_path = os.path.relpath(audio_path, BASE_DRIVE_PATH)
        dataset_records.append({
            "audio_path": relative_path,
            "malayalam_transcription": malayalam_text,
            "english_translation": english_text
        })

    if not dataset_records: return

    print(f"\nAppending {len(dataset_records)} new records to JSONL file...")
    with open(metadata_output_path, 'a', encoding='utf-8') as f:
        for record in dataset_records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f"🎉 Successfully appended metadata to: {metadata_output_path}")

In [ ]:
# --- 7. Run the Full Automation Loop ---
import pandas as pd
import os

video_counter = 0

while True:
    # --- Step A: Check for and setup the tracking file ---
    if not os.path.exists(TRACKING_FILE_PATH):
        print("Tracking file not found. Creating a new one...")
        video_urls = get_playlist_video_urls(PLAYLIST_URL)
        if not video_urls:
            print("🚨 No URLs found in the playlist. Halting.")
            break
        df = pd.DataFrame({'URL': video_urls, 'Status': ''})
        df.to_excel(TRACKING_FILE_PATH, index=False)
        print(f"✅ Tracking file created at {TRACKING_FILE_PATH}")

    # --- Step B: Find the next video to process ---
    df = pd.read_excel(TRACKING_FILE_PATH)
    next_video = df[df['Status'] != 'Done'].iloc[0] if not df[df['Status'] != 'Done'].empty else None

    if next_video is None:
        print("\n🎉 All videos in the playlist have been processed! Congratulations! 🎉")
        break

    current_video_url = next_video['URL']
    print(f"\n▶️ Processing next video: {current_video_url}")

    # --- Step C: Run the dataset creation pipeline for the selected video ---
    try:
        print("\n--- Step 1 of 5: Downloading Audio ---")
        downloaded_mp3_path = download_audio_from_youtube(current_video_url, RAW_AUDIO_PATH)

        if downloaded_mp3_path and os.path.exists(downloaded_mp3_path):
            print("\n--- Step 2 of 5: Trimming Audio ---")
            trimmed_path = trim_audio(downloaded_mp3_path, 30)

            if trimmed_path:
                print("\n--- Step 3 of 5: Separating Vocals from Music ---")
                vocals_only_path = separate_vocals_with_demucs(trimmed_path, DEMUCS_OUTPUT_DIR)

                if vocals_only_path and os.path.exists(vocals_only_path):
                    print("\n--- Step 4 of 5: Preprocessing and Splitting Audio ---")
                    audio_chunks = preprocess_and_split_audio(vocals_only_path, AUDIO_CHUNKS_FOLDER)

                    if audio_chunks:
                        print("\n--- Step 5 of 5: Creating Dataset ---")
                        create_dataset_pipeline(audio_chunks, METADATA_FILE_PATH)

                        # --- Step D: Mark video as 'Done' in the tracking file ---
                        df.loc[df['URL'] == current_video_url, 'Status'] = 'Done'
                        df.to_excel(TRACKING_FILE_PATH, index=False)
                        print(f"✅ Marked {current_video_url} as 'Done'.")
                        video_counter += 1
                    else:
                        raise Exception("No audio chunks were created.")
                else:
                    raise Exception("Music separation failed.")
            else:
                raise Exception("Audio trimming failed.")
        else:
            raise Exception("Audio download failed.")

    except Exception as e:
        print(f"🚨 An error occurred while processing {current_video_url}: {e}")
        print("Marking as 'Failed' and moving to the next video.")
        df.loc[df['URL'] == current_video_url, 'Status'] = 'Failed'
        df.to_excel(TRACKING_FILE_PATH, index=False)
        continue

    # --- Step E: Check if it's time to pause ---
    if video_counter > 0 and video_counter % 5 == 0:
        if user_halt_check():
            break


▶️ Processing next video: https://www.youtube.com/watch?v=d5UCAkxBhqQ

--- Step 1 of 5: Downloading Audio ---
✅ Audio downloaded successfully to /content/drive/MyDrive/Automated_Malayalam_Dataset/raw_downloaded_audio.mp3

--- Step 2 of 5: Trimming Audio ---
✂️ Trimming last 30 seconds from the audio...
✅ Audio trimmed successfully. New duration: 1160.87 seconds.

--- Step 3 of 5: Separating Vocals from Music ---
🎶 Separating vocals from music with Demucs... (This may take a few minutes)
Important: the default model was recently changed to `htdemucs` the latest Hybrid Transformer Demucs model. In some cases, this model can actually perform worse than previous models. To get back the old default model use `-n mdx_extra_q`.
Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/955717e8-8726e21a.th" to /root/.cache/torch/hub/checkpoints/955717e8-8726e21a.th
100% 80.2M/80.2M [00:00<00:00, 173MB/s]
Selected model is a bag of 1 models. You will see that many progress bars pe

Saving Chunks:   0%|          | 0/161 [00:00<?, ?it/s]

✅ Saved 145 audio chunks to /content/drive/MyDrive/Automated_Malayalam_Dataset/audio

--- Step 5 of 5: Creating Dataset ---

🤖 Starting transcription and translation pipeline...


Processing Files:   0%|          | 0/145 [00:00<?, ?it/s]

File: malayalam_speech_18124.wav -> Transcription failed. Deleting file.
File: malayalam_speech_18216.wav -> Transcription failed. Deleting file.
Translation failed for '50 വർഷം പ്രായമുള്ള ഒരു പുളിമരവും പുരയിട': ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

Appending 143 new records to JSONL file...
🎉 Successfully appended metadata to: /content/drive/MyDrive/Automated_Malayalam_Dataset/metadata.jsonl
✅ Marked https://www.youtube.com/watch?v=d5UCAkxBhqQ as 'Done'.

▶️ Processing next video: https://www.youtube.com/watch?v=Svl2B584XL0

--- Step 1 of 5: Downloading Audio ---
✅ Audio downloaded successfully to /content/drive/MyDrive/Automated_Malayalam_Dataset/raw_downloaded_audio.mp3

--- Step 2 of 5: Trimming Audio ---
✂️ Trimming last 30 seconds from the audio...
✅ Audio trimmed successfully. New duration: 929.60 seconds.

--- Step 3 of 5: Separating Vocals from Music ---
🎶 Separating vocals from music with Demucs... (This may take a few m

...

Translation failed for 'അങ്ങനെ അംഗീകാരങ്ങളുടെ പട്ടിക ഇവിടെ പറഞ്ഞുതീർക്കാൻ പറ്റുന്നതല്ല അതിലെല്ലാമുപരി': ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

Appending 210 new records to JSONL file...
🎉 Successfully appended metadata to: /content/drive/MyDrive/Automated_Malayalam_Dataset/metadata.jsonl
✅ Marked https://www.youtube.com/watch?v=G8B5e9uH-v4 as 'Done'.

▶️ Processing next video: https://www.youtube.com/watch?v=ssDX2vHrOLE

--- Step 1 of 5: Downloading Audio ---
✅ Audio downloaded successfully to /content/drive/MyDrive/Automated_Malayalam_Dataset/raw_downloaded_audio.mp3

--- Step 2 of 5: Trimming Audio ---
✂️ Trimming last 30 seconds from the audio...
✅ Audio trimmed successfully. New duration: 1437.93 seconds.

--- Step 3 of 5: Separating Vocals from Music ---
🎶 Separating vocals from music with Demucs... (This may take a few minutes)
Important: the default model was recently changed to `htdemucs` the latest Hybrid Transformer Demuc

Saving Chunks:   0%|          | 0/215 [00:00<?, ?it/s]

✅ Saved 197 audio chunks to /content/drive/MyDrive/Automated_Malayalam_Dataset/audio

--- Step 5 of 5: Creating Dataset ---

🤖 Starting transcription and translation pipeline...


Processing Files:   0%|          | 0/197 [00:00<?, ?it/s]

Translation failed for 'അങ്ങനെ തങ്ങളുടെ പ്രദേശത്തേക്ക് മാത്രമായ കാലാവസ്ഥാപ്രവചനം സാധ്യമായത് സന്തോഷത്തിലാണ് മയിൽ പഞ്ചായ': ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

Appending 197 new records to JSONL file...
🎉 Successfully appended metadata to: /content/drive/MyDrive/Automated_Malayalam_Dataset/metadata.jsonl
✅ Marked https://www.youtube.com/watch?v=ssDX2vHrOLE as 'Done'.

🎉 All videos in the playlist have been processed! Congratulations! 🎉


##Cleanup Cells

In [ ]:
# --- 8. Renumber and Finalize Dataset ---
# This cell re-numbers all audio files and their corresponding entries in the
# metadata.jsonl file to be sequential, starting from 1.
# This is useful after processing a full playlist to ensure clean, sequential numbering.

import json
import os
import re

def renumber_dataset():
    print("🔢 Starting re-numbering process...")
    try:
        if not os.path.exists(METADATA_FILE_PATH):
            raise FileNotFoundError(f"Metadata file not found at {METADATA_FILE_PATH}.")

        records = []
        with open(METADATA_FILE_PATH, 'r', encoding='utf-8') as f:
            for line in f:
                records.append(json.loads(line))

        def get_filenumber(path):
            match = re.search(r'_(\d+)\.wav$', path)
            return int(match.group(1)) if match else 0

        records.sort(key=lambda r: get_filenumber(r['audio_path']))
        print(f"Found {len(records)} records to re-number.")

        updated_records = []
        for new_index, record in enumerate(records, start=1):
            old_relative_path = record['audio_path']
            old_full_path = os.path.join(BASE_DRIVE_PATH, old_relative_path)
            new_relative_path = f"audio/malayalam_speech_{new_index}.wav"
            new_full_path = os.path.join(BASE_DRIVE_PATH, new_relative_path)

            if os.path.exists(old_full_path) and old_full_path != new_full_path:
                os.rename(old_full_path, new_full_path)
            elif not os.path.exists(old_full_path):
                print(f"  - Warning: Could not find {old_full_path}. Skipping.")
                continue

            record['audio_path'] = new_relative_path
            updated_records.append(record)

        with open(METADATA_FILE_PATH, 'w', encoding='utf-8') as f:
            for rec in updated_records:
                f.write(json.dumps(rec, ensure_ascii=False) + '\n')

        print(f"\n✅ Renumbering complete. Updated {len(updated_records)} audio files and metadata records.")
    except Exception as e:
        print(f"🚨 An error occurred during re-numbering: {e}")

# You can run this function by calling it after the main automation loop is finished.
# renumber_dataset()

In [ ]:
#Remove Orphaned files(audio files not in metadata)
import os
import json

# --- Configuration ---
# These paths should match the ones in your main notebook.
BASE_DRIVE_PATH = "/content/drive/MyDrive/Automated_Malayalam_Dataset"
AUDIO_CHUNKS_FOLDER = os.path.join(BASE_DRIVE_PATH, "audio")
METADATA_FILE_PATH = os.path.join(BASE_DRIVE_PATH, "metadata.jsonl")


def clean_orphaned_audio_files():
    """
    Scans the audio directory and removes any .wav files that
    do not have a corresponding entry in the metadata.jsonl file.
    """
    print("--- 🧹 Starting Audio Cleanup ---")

    # 1. Read all valid audio paths from the metadata file
    try:
        print(f"Reading metadata from: {METADATA_FILE_PATH}")
        valid_audio_paths = set()
        with open(METADATA_FILE_PATH, 'r', encoding='utf-8') as f:
            for line in f:
                # Ensure the line is not empty
                if line.strip():
                    record = json.loads(line)
                    if 'audio_path' in record:
                        valid_audio_paths.add(record['audio_path'])

        if not valid_audio_paths:
            print("🚨 Warning: No valid audio paths found in metadata. Aborting to be safe.")
            return

        print(f"Found {len(valid_audio_paths)} records in the metadata file.")
    except FileNotFoundError:
        print(f"🚨 Error: Metadata file not found at {METADATA_FILE_PATH}. Aborting.")
        return
    except json.JSONDecodeError as e:
        print(f"🚨 Error: Could not read the metadata file. It might be corrupted. {e}")
        return

    # 2. Scan the audio directory to find all actual .wav files
    print(f"Scanning for .wav files in: {AUDIO_CHUNKS_FOLDER}")
    try:
        all_audio_files = os.listdir(AUDIO_CHUNKS_FOLDER)
    except FileNotFoundError:
        print(f"🚨 Error: Audio chunks folder not found at {AUDIO_CHUNKS_FOLDER}. Aborting.")
        return

    orphaned_files_to_delete = []
    for filename in all_audio_files:
        if filename.endswith('.wav'):
            # Construct the relative path as it appears in the metadata
            relative_path = os.path.join("audio", filename)

            # If the file on disk is NOT in our set of valid paths, it's an orphan
            if relative_path not in valid_audio_paths:
                full_path = os.path.join(AUDIO_CHUNKS_FOLDER, filename)
                orphaned_files_to_delete.append(full_path)

    # 3. Report findings and ask for user confirmation before deleting
    if not orphaned_files_to_delete:
        print("\n✅ No orphaned audio files found. Your dataset is clean!")
        return

    print(f"\nFound {len(orphaned_files_to_delete)} orphaned audio files that are not in the metadata.")
    print("Here are the first few examples:")
    for file_path in orphaned_files_to_delete[:10]:
        print(f"  - {os.path.basename(file_path)}")
    if len(orphaned_files_to_delete) > 10:
        print("  ...")

    try:
        confirm = input("\nDo you want to permanently delete these files? (Type 'yes' to confirm): ").lower()
    except EOFError:
        print("\nCould not read input. Aborting deletion.")
        return

    # 4. Delete the files if confirmed
    if confirm == 'yes':
        print("Deleting files...")
        deleted_count = 0
        for file_path in orphaned_files_to_delete:
            try:
                os.remove(file_path)
                deleted_count += 1
            except OSError as e:
                print(f"  - Could not delete {os.path.basename(file_path)}: {e}")

        print(f"\n✅ Cleanup complete. Successfully deleted {deleted_count} orphaned files.")
    else:
        print("\nDeletion cancelled by user. No files were changed.")


# --- Run the cleanup function ---
clean_orphaned_audio_files()

In [ ]:
import os
from pydub import AudioSegment
from tqdm.notebook import tqdm
import traceback

# --- Configuration ---
# This path should match the one in your main notebook.
BASE_DRIVE_PATH = "/content/drive/MyDrive/Automated_Malayalam_Dataset"
AUDIO_CHUNKS_FOLDER = os.path.join(BASE_DRIVE_PATH, "audio")

def calculate_total_duration():
    """
    Scans the audio directory, calculates the total duration of all
    .wav files, and prints the result.
    """
    print("--- ⏱️ Calculating Total Dataset Duration ---")

    if not os.path.exists(AUDIO_CHUNKS_FOLDER):
        print(f"🚨 Error: Audio folder not found at {AUDIO_CHUNKS_FOLDER}")
        return

    total_milliseconds = 0
    corrupted_files = []

    # Get a list of all .wav files to process
    wav_files = [f for f in os.listdir(AUDIO_CHUNKS_FOLDER) if f.endswith('.wav')]

    if not wav_files:
        print("No .wav files found in the audio folder.")
        return

    print(f"Found {len(wav_files)} audio files. Analyzing duration...")

    # Loop through all files with a progress bar
    for filename in tqdm(wav_files, desc="Processing files"):
        file_path = os.path.join(AUDIO_CHUNKS_FOLDER, filename)
        try:
            audio = AudioSegment.from_wav(file_path)
            total_milliseconds += len(audio)
        except Exception as e:
            # If a file is corrupted, note it and continue
            corrupted_files.append(filename)
            # print(f"\nWarning: Could not process file '{filename}'. It may be corrupted. Error: {e}")
            # traceback.print_exc()


    # --- Convert Milliseconds to a Human-Readable Format ---
    total_seconds = total_milliseconds / 1000
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = int(total_seconds % 60)

    print("\n" + "="*40)
    print(f"✅ Total Dataset Duration: {hours} hours, {minutes} minutes, {seconds} seconds.")
    print("="*40)

    if corrupted_files:
        print(f"\n🚨 Found {len(corrupted_files)} files that could not be read and were excluded:")
        for cf in corrupted_files[:5]: # Print first 5 corrupted files
             print(f"  - {cf}")
        if len(corrupted_files) > 5:
             print("  ...")


# --- Run the calculation ---
calculate_total_duration()

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


--- ⏱️ Calculating Total Dataset Duration ---
Found 24807 audio files. Analyzing duration...


Processing files:   0%|          | 0/24807 [00:00<?, ?it/s]


✅ Total Dataset Duration: 35 hours, 42 minutes, 47 seconds.
